<a href="https://colab.research.google.com/github/emrealty/dta-transformer/blob/main/_Full_4x4x4x4_Ablation_Runner_Positional_Encoding_4_Normalization_4_FFN_Type_4_Cross_Attn_Type_4_%3D_256_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
###########################################
# Full 4x4x4x4 Ablation Runner
# - Positional Encoding: 4
# - Normalization: 4
# - FFN Type: 4
# - Cross-Attn Type: 4
# = 256 model
###########################################

######################
# [BÖLÜM 1: Ortam Kurulumu, Paketler]
######################
print("===== 1) Ortam Kurulumu ve Paketler =====")

from google.colab import drive
drive.mount('/content/drive')

!pip install rdkit-pypi -q
!pip install torch -q
!pip install transformers -q
!pip install scikit-learn -q
!pip install scipy -q

import math
import copy
import random
import re
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, average_precision_score
from math import sqrt
from scipy.stats import pearsonr

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Kullanılan cihaz:", device)

######################
# [BÖLÜM 2: Seed Ayarı]
######################
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)

######################
# [BÖLÜM 3: Veri Yükleme ve İnceleme]
######################
print("\n===== 2) Veri Okuma ve İnceleme =====")

# Yolunu kendi dosyana göre güncelle
file_path = '/content/drive/MyDrive/TransformDTA/davis_cleaned.csv'

df = pd.read_csv(file_path, sep=',', header=0)
df.rename(columns={
    "Compound_ID": "ID",
    "Protein_ID": "Target",
    "SMILES": "SMILES",
    "Protein_Sequence": "Sequence",
    "Label": "Label"
}, inplace=True)

print("DataFrame ilk 5 satır:")
print(df.head())
print("\nVeri boyutu:", df.shape)
print("Label istatistikleri:")
print(df['Label'].describe())

######################
# [BÖLÜM 4: Train / Val / Test Split]
######################
print("\n===== 3) Train-Val-Test Split =====")

train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

print("Train:", train_df.shape, "Val:", val_df.shape, "Test:", test_df.shape)

######################
# [BÖLÜM 5: Tokenizasyon ve Dataset/DataLoader]
######################
print("\n===== 4) Tokenizasyon, Dataset, DataLoader =====")

def tokenize_protein(seq):
    return list(seq.strip())

_SMILES_REGEX = re.compile(
    r"\[\w+\]|"       # [NH3+], [O-] vb
    r"\%\d{2}|"       # iki haneli ring
    r"Br|Cl|"         # iki harfli atomlar
    r"Si|Se|Na|Li|Mg|Ca|Zn|Fe|Cu|Mn|Al|Ag|Sn|Hg|Pb|Bi|Ne|He|Ar|Kr|Xe|"
    r"@@?|"           # stereo
    r"=|#|"           # bağlar
    r"\(|\)|"         # parantez
    r"\.|"            # nokta
    r"\d|"            # rakamlar
    r"[A-Za-z]|"      # tek harfli atomlar
    r"\+|\-|\*|\/|\\" # yükler ve semboller
)

def tokenize_smiles(smi: str):
    return _SMILES_REGEX.findall(smi.strip())

# test
test_smiles = "CCCCl"
print("\nSMILES Tokenizer Testi:")
print("  Input:", test_smiles)
print("  Tokens:", tokenize_smiles(test_smiles))
print("  Beklenen: ['C','C','C','Cl']")

test_smiles2 = "c1ccccc1Br"
print("  Input:", test_smiles2)
print("  Tokens:", tokenize_smiles(test_smiles2))

# vocab
all_prot_tokens = set()
for seq in train_df['Sequence']:
    for t in tokenize_protein(seq):
        all_prot_tokens.add(t)

all_smi_tokens = set()
for smi in train_df['SMILES']:
    for t in tokenize_smiles(smi):
        all_smi_tokens.add(t)

prot_special_tokens = ['<pad>', '<unk>', '<cls>', '<sep>']
smi_special_tokens  = ['<pad>', '<unk>', '<cls>', '<sep>']

prot_vocab_list = prot_special_tokens + sorted(list(all_prot_tokens))
smi_vocab_list  = smi_special_tokens  + sorted(list(all_smi_tokens))

protein_vocab = {token: idx for idx, token in enumerate(prot_vocab_list)}
smiles_vocab  = {token: idx for idx, token in enumerate(smi_vocab_list)}

print(f"\n✅ Protein vocab size: {len(protein_vocab)}")
print(f"✅ SMILES vocab size:  {len(smiles_vocab)}")
print("   'Cl' vocabda mı?", 'Cl' in smiles_vocab)
print("   'Br' vocabda mı?", 'Br' in smiles_vocab)

class DTADataset(Dataset):
    def __init__(self, df, protein_vocab, smiles_vocab,
                 max_prot_len=1000, max_smi_len=100):
        self.df = df.reset_index(drop=True)
        self.protein_vocab = protein_vocab
        self.smiles_vocab  = smiles_vocab
        self.max_prot_len  = max_prot_len
        self.max_smi_len   = max_smi_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        seq = row['Sequence']
        smi = row['SMILES']
        label = float(row['Label'])

        prot_toks = tokenize_protein(seq)[:self.max_prot_len]
        smi_toks  = tokenize_smiles(smi)[:self.max_smi_len]

        prot_ids = [self.protein_vocab.get(t, self.protein_vocab['<unk>']) for t in prot_toks]
        smi_ids  = [self.smiles_vocab.get(t, self.smiles_vocab['<unk>'])   for t in smi_toks]

        prot_pad_len = self.max_prot_len - len(prot_ids)
        smi_pad_len  = self.max_smi_len  - len(smi_ids)

        prot_ids += [self.protein_vocab['<pad>']] * prot_pad_len
        smi_ids  += [self.smiles_vocab['<pad>']]  * smi_pad_len

        return {
            'protein_input': torch.tensor(prot_ids, dtype=torch.long),
            'smiles_input':  torch.tensor(smi_ids,  dtype=torch.long),
            'label':         torch.tensor(label,    dtype=torch.float)
        }

batch_size = 32  # istersen 64 yapabilirsin
train_dataset = DTADataset(train_df, protein_vocab, smiles_vocab)
val_dataset   = DTADataset(val_df,   protein_vocab, smiles_vocab)
test_dataset  = DTADataset(test_df,  protein_vocab, smiles_vocab)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)

print(f"\ntrain: {len(train_dataset)}, val: {len(val_dataset)}, test: {len(test_dataset)}")

######################
# [BÖLÜM 6: Metrikler ve Train/Eval Fonksiyonları]
######################
def train_one_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    for batch in dataloader:
        prot_in = batch['protein_input'].to(device)
        smi_in  = batch['smiles_input'].to(device)
        labels  = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model(prot_in, smi_in)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(dataloader)

@torch.no_grad()
def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    preds = []
    trues = []
    for batch in dataloader:
        prot_in = batch['protein_input'].to(device)
        smi_in  = batch['smiles_input'].to(device)
        labels  = batch['label'].to(device)

        outputs = model(prot_in, smi_in)
        loss = criterion(outputs, labels)
        total_loss += loss.item()
        preds.extend(outputs.cpu().numpy())
        trues.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(dataloader)
    return avg_loss, np.array(preds), np.array(trues)

def concordance_index(y_true, y_pred, max_points=2000):
    """
    CI hesaplıyor, hız için büyük test setlerinde max_points kadar örnek subsample ediyor.
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    n = len(y_true)
    if n > max_points:
        idx = np.random.choice(n, size=max_points, replace=False)
        y_true = y_true[idx]
        y_pred = y_pred[idx]
        n = max_points

    n_pairs, n_conc = 0, 0
    for i in range(n):
        for j in range(i + 1, n):
            if y_true[i] == y_true[j]:
                continue
            n_pairs += 1
            if (y_true[i] > y_true[j] and y_pred[i] > y_pred[j]) or \
               (y_true[i] < y_true[j] and y_pred[i] < y_pred[j]):
                n_conc += 1
    if n_pairs == 0:
        return 0.0
    return n_conc / n_pairs

def rm2_score(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    r, _ = pearsonr(y_true, y_pred)
    r_sq = r**2

    denom = np.sum(y_true*y_true) - len(y_true)*(np.mean(y_true)**2)
    if abs(denom) < 1e-8:
        return 0.0

    slope = (np.sum(y_true*y_pred) - len(y_true)*np.mean(y_true)*np.mean(y_pred)) / denom
    intercept = np.mean(y_pred) - slope*np.mean(y_true)
    y_pred_reg = slope*y_true + intercept
    r0, _ = pearsonr(y_true, y_pred_reg)
    r0_sq = r0**2
    return r_sq * (1 - np.sqrt(abs(r_sq - r0_sq)))

def aupr_score(y_true, y_pred, threshold=7.0):
    labels_bin = np.array([1 if v > threshold else 0 for v in y_true])
    return average_precision_score(labels_bin, y_pred)

######################
# [BÖLÜM 7: Normalization, RoPE, ALiBi ve Attention Modülleri]
######################
print("\n===== 5) Model Bileşenleri (Norm, RoPE, ALiBi, Attention) =====")

class RMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-8):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d_model))
        self.eps = eps

    def forward(self, x):
        rms = x.pow(2).mean(dim=-1, keepdim=True).add(self.eps).rsqrt()
        x_norm = x * rms
        return x_norm * self.weight

def build_norm(norm_type: str, d_model: int):
    if norm_type == 'ln':
        return nn.LayerNorm(d_model)
    elif norm_type == 'rms':
        return RMSNorm(d_model)
    else:
        raise ValueError(f"Unknown norm_type: {norm_type}")

class RotaryPositionEmbedding(nn.Module):
    """
    RoPE: Q ve K'ya başlık içi döndürme uygular.
    """
    def __init__(self, head_dim, base=10000):
        super().__init__()
        assert head_dim % 2 == 0, "RoPE head_dim çift olmalı."
        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)

    def _build_cos_sin(self, seq_len, device):
        t = torch.arange(seq_len, device=device).float()
        freqs = torch.einsum('i,j->ij', t, self.inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        return emb.cos()[None, None, :, :], emb.sin()[None, None, :, :]

    def apply_rotary(self, x, cos, sin):
        # x: [B, H, S, D]
        x1, x2 = x[..., ::2], x[..., 1::2]
        x_rot = torch.stack([-x2, x1], dim=-1).reshape_as(x)
        return (x * cos) + (x_rot * sin)

    def forward(self, q, k):
        B, H, Q, D = q.shape
        _, _, K, _ = k.shape
        cos_q, sin_q = self._build_cos_sin(Q, q.device)
        cos_k, sin_k = self._build_cos_sin(K, k.device)
        q = self.apply_rotary(q, cos_q, sin_q)
        k = self.apply_rotary(k, cos_k, sin_k)
        return q, k

def _get_alibi_slopes(n_heads: int):
    def get_slopes_power_of_2(n):
        start = 2**(-8.0 / n)
        ratio = start
        return [start * (ratio**i) for i in range(n)]
    if math.log2(n_heads).is_integer():
        slopes = get_slopes_power_of_2(n_heads)
    else:
        closest = 2**math.floor(math.log2(n_heads))
        slopes = get_slopes_power_of_2(closest)
        extra = get_slopes_power_of_2(2*closest)[0::2][:n_heads - closest]
        slopes += extra
    return torch.tensor(slopes)

def build_alibi_bias(n_heads, q_len, k_len, device):
    q_pos = torch.arange(q_len, device=device)[:, None]  # [Q,1]
    k_pos = torch.arange(k_len, device=device)[None, :]  # [1,K]
    dist = (q_pos - k_pos).abs().float()                 # [Q,K]
    slopes = _get_alibi_slopes(n_heads).to(device)       # [H]
    bias = -slopes[:, None, None] * dist[None, :, :]     # [H,Q,K]
    return bias.unsqueeze(0)                              # [1,H,Q,K]

class MultiheadAttentionWithPE(nn.Module):
    """
    Self- / cross-attention:
    pe_type: 'none' | 'rope' | 'alibi'
    """
    def __init__(self, d_model, nhead, dropout=0.1, pe_type='none'):
        super().__init__()
        assert d_model % nhead == 0
        self.d_model = d_model
        self.nhead = nhead
        self.head_dim = d_model // nhead
        self.pe_type = pe_type

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        if self.pe_type == 'rope':
            self.rope = RotaryPositionEmbedding(self.head_dim)
        else:
            self.rope = None

    def _shape(self, x, B):
        return x.view(B, -1, self.nhead, self.head_dim).transpose(1, 2)  # [B,H,S,D]

    def forward(self, query, key, value, key_padding_mask=None):
        B, Q, _ = query.shape
        K = key.size(1)

        q = self._shape(self.q_proj(query), B)  # [B,H,Q,D]
        k = self._shape(self.k_proj(key),   B)  # [B,H,K,D]
        v = self._shape(self.v_proj(value), B)  # [B,H,K,D]

        if self.pe_type == 'rope' and self.rope is not None:
            q, k = self.rope(q, k)

        scale = 1.0 / math.sqrt(self.head_dim)
        scores = torch.matmul(q, k.transpose(-2, -1)) * scale  # [B,H,Q,K]

        if self.pe_type == 'alibi':
            alibi = build_alibi_bias(self.nhead, Q, K, device=query.device)  # [1,H,Q,K]
            scores = scores + alibi

        if key_padding_mask is not None:
            mask = key_padding_mask[:, None, None, :].to(torch.bool)  # [B,1,1,K]
            scores = scores.masked_fill(mask, float('-inf'))

        attn = torch.softmax(scores, dim=-1)
        attn = self.attn_dropout(attn)

        ctx = torch.matmul(attn, v)  # [B,H,Q,D]
        ctx = ctx.transpose(1, 2).contiguous().view(B, Q, self.d_model)  # [B,Q,D]
        out = self.out_proj(ctx)
        out = self.resid_dropout(out)
        return out

######################
# FFN (ReLU / GELU / SwiGLU / GeGLU)
######################
class FFN(nn.Module):
    def __init__(self, d_model=256, dim_feedforward=1024, dropout=0.1, ffn_type="relu"):
        super().__init__()
        self.ffn_type = ffn_type.lower()
        self.dropout = nn.Dropout(dropout)

        if self.ffn_type in ["relu", "gelu"]:
            self.linear1 = nn.Linear(d_model, dim_feedforward)
            self.linear2 = nn.Linear(dim_feedforward, d_model)
            self.act = nn.ReLU() if self.ffn_type == "relu" else nn.GELU()
        elif self.ffn_type in ["swiglu", "geglu"]:
            h_glu = int(dim_feedforward * 2 / 3)
            self.linear_v = nn.Linear(d_model, h_glu)
            self.linear_g = nn.Linear(d_model, h_glu)
            self.out_proj = nn.Linear(h_glu, d_model)
            self.act = nn.SiLU() if self.ffn_type == "swiglu" else nn.GELU()
        else:
            raise ValueError(f"Geçersiz ffn_type: {ffn_type}")

    def forward(self, x):
        if self.ffn_type in ["relu", "gelu"]:
            return self.linear2(self.dropout(self.act(self.linear1(x))))
        else:
            v = self.linear_v(x)
            g = self.act(self.linear_g(x))
            return self.out_proj(self.dropout(v * g))

######################
# Encoder Bloğu (Pre/Post Norm + LN/RMS + FFN tipi + RoPE/ALiBi)
######################
class TransformerEncoderBlockGeneral(nn.Module):
    def __init__(self, d_model=256, nhead=8, dim_feedforward=1024, dropout=0.1,
                 ffn_type="relu", pre_norm=False, norm_type='ln', attn_pe='none'):
        super().__init__()
        self.pre_norm = pre_norm
        self.self_attn = MultiheadAttentionWithPE(d_model, nhead, dropout, pe_type=attn_pe)
        self.ffn = FFN(d_model, dim_feedforward, dropout, ffn_type=ffn_type)
        self.norm1 = build_norm(norm_type, d_model)
        self.norm2 = build_norm(norm_type, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, key_padding_mask=None):
        # Self-Attn
        if self.pre_norm:
            x_norm = self.norm1(x)
            attn_out = self.self_attn(x_norm, x_norm, x_norm, key_padding_mask=key_padding_mask)
            x = x + self.dropout(attn_out)
        else:
            attn_out = self.self_attn(x, x, x, key_padding_mask=key_padding_mask)
            x = self.norm1(x + self.dropout(attn_out))

        # FFN
        if self.pre_norm:
            x_norm = self.norm2(x)
            ff_out = self.ffn(x_norm)
            x = x + self.dropout(ff_out)
        else:
            ff_out = self.ffn(x)
            x = self.norm2(x + self.dropout(ff_out))

        return x

######################
# Cross-Attention Blokları (Standard / Gated / GQA)
######################
class BiDirectionalCrossAttention(nn.Module):
    """
    Standard veya Gated çift yönlü cross-attention.
    attn_pe: 'none' | 'rope' | 'alibi'
    """
    def __init__(self, d_model=256, nhead=8, dropout=0.1,
                 attn_pe='none', pre_norm=False, norm_type='ln', gated=False):
        super().__init__()
        self.pre_norm = pre_norm
        self.gated = gated

        self.p2s = MultiheadAttentionWithPE(d_model, nhead, dropout, pe_type=attn_pe)
        self.s2p = MultiheadAttentionWithPE(d_model, nhead, dropout, pe_type=attn_pe)
        self.dropout = nn.Dropout(dropout)

        self.p_norm1 = build_norm(norm_type, d_model)
        self.s_norm1 = build_norm(norm_type, d_model)
        self.p_norm2 = build_norm(norm_type, d_model)
        self.s_norm2 = build_norm(norm_type, d_model)

        if self.gated:
            self.gate_p = nn.Parameter(torch.zeros(d_model))
            self.gate_s = nn.Parameter(torch.zeros(d_model))

    def _apply_gate(self, x, gate_param):
        g = torch.sigmoid(gate_param).view(1, 1, -1)
        return x * g

    def forward(self, p, s, p_mask=None, s_mask=None):
        # p -> s yönü
        if self.pre_norm:
            p_in = self.p_norm1(p)
            s_in = self.s_norm1(s)
            p2 = self.p2s(p_in, s_in, s_in, key_padding_mask=s_mask)
            if self.gated:
                p = p + self._apply_gate(self.dropout(p2), self.gate_p)
            else:
                p = p + self.dropout(p2)
        else:
            p2 = self.p2s(p, s, s, key_padding_mask=s_mask)
            if self.gated:
                p = p + self._apply_gate(self.dropout(p2), self.gate_p)
            else:
                p = p + self.dropout(p2)
            p = self.p_norm1(p)

        # s -> p yönü
        if self.pre_norm:
            s_in = self.s_norm2(s)
            p_in = self.p_norm2(p)
            s2 = self.s2p(s_in, p_in, p_in, key_padding_mask=p_mask)
            if self.gated:
                s = s + self._apply_gate(self.dropout(s2), self.gate_s)
            else:
                s = s + self.dropout(s2)
        else:
            s2 = self.s2p(s, p, p, key_padding_mask=p_mask)
            if self.gated:
                s = s + self._apply_gate(self.dropout(s2), self.gate_s)
            else:
                s = s + self.dropout(s2)
            s = self.s_norm2(s)

        return p, s

class GroupedQueryCrossAttentionBlockGeneral(nn.Module):
    """
    GQA tabanlı çift yönlü cross-attention.
    nhead: Q başlık sayısı
    n_kv_heads: KV başlık sayısı (ör. 8Q, 4KV)
    """
    def __init__(self, d_model=256, nhead=8, n_kv_heads=4, dropout=0.1,
                 attn_pe='none', pre_norm=False, norm_type='ln'):
        super().__init__()
        assert nhead % n_kv_heads == 0, "nhead, n_kv_heads'in katı olmalı."
        self.d_model = d_model
        self.nhead = nhead
        self.n_kv = n_kv_heads
        self.hdim = d_model // nhead
        self.group = nhead // n_kv_heads

        self.pre_norm = pre_norm
        self.pe_type = attn_pe

        # p->s projeksiyonları
        self.q_proj_ps = nn.Linear(d_model, nhead * self.hdim)
        self.k_proj_ps = nn.Linear(d_model, n_kv_heads * self.hdim)
        self.v_proj_ps = nn.Linear(d_model, n_kv_heads * self.hdim)
        self.out_ps = nn.Linear(d_model, d_model)

        # s->p projeksiyonları
        self.q_proj_sp = nn.Linear(d_model, nhead * self.hdim)
        self.k_proj_sp = nn.Linear(d_model, n_kv_heads * self.hdim)
        self.v_proj_sp = nn.Linear(d_model, n_kv_heads * self.hdim)
        self.out_sp = nn.Linear(d_model, d_model)

        self.attn_drop = nn.Dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)

        self.p_norm1 = build_norm(norm_type, d_model)
        self.s_norm1 = build_norm(norm_type, d_model)
        self.p_norm2 = build_norm(norm_type, d_model)
        self.s_norm2 = build_norm(norm_type, d_model)

        if self.pe_type == 'rope':
            self.rope = RotaryPositionEmbedding(self.hdim)
        else:
            self.rope = None

    def _cross(self, q_in, kv_in, q_mask, kv_mask, q_proj, k_proj, v_proj, out_proj):
        B, Lq, D = q_in.shape
        Lk = kv_in.size(1)

        q = q_proj(q_in).view(B, Lq, self.nhead, self.hdim).transpose(1, 2)   # [B,H,Lq,Hd]
        k = k_proj(kv_in).view(B, Lk, self.n_kv, self.hdim).transpose(1, 2)   # [B,H_kv,Lk,Hd]
        v = v_proj(kv_in).view(B, Lk, self.n_kv, self.hdim).transpose(1, 2)   # [B,H_kv,Lk,Hd]

        if self.n_kv != self.nhead:
            k = k.repeat_interleave(self.group, dim=1)
            v = v.repeat_interleave(self.group, dim=1)

        if self.pe_type == 'rope' and self.rope is not None:
            q, k = self.rope(q, k)

        attn_scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.hdim)  # [B,H,Lq,Lk]

        if self.pe_type == 'alibi':
            alibi = build_alibi_bias(self.nhead, Lq, Lk, device=q_in.device)  # [1,H,Lq,Lk]
            attn_scores = attn_scores + alibi

        if kv_mask is not None:
            attn_scores = attn_scores.masked_fill(kv_mask[:, None, None, :], -1e9)

        attn = torch.softmax(attn_scores, dim=-1)
        attn = self.attn_drop(attn)

        out = torch.matmul(attn, v)  # [B,H,Lq,Hd]
        out = out.transpose(1, 2).contiguous().view(B, Lq, self.d_model)
        out = out_proj(out)
        out = self.resid_drop(out)
        return out

    def forward(self, p, s, p_mask=None, s_mask=None):
        # p (query) -> s (key/value)
        if self.pre_norm:
            p_in = self.p_norm1(p)
            s_in = self.s_norm1(s)
            p2 = self._cross(p_in, s_in, p_mask, s_mask,
                             self.q_proj_ps, self.k_proj_ps, self.v_proj_ps, self.out_ps)
            p = p + p2
        else:
            p2 = self._cross(p, s, p_mask, s_mask,
                             self.q_proj_ps, self.k_proj_ps, self.v_proj_ps, self.out_ps)
            p = self.p_norm1(p + p2)

        # s (query) -> p (key/value)
        if self.pre_norm:
            s_in = self.s_norm2(s)
            p_in = self.p_norm2(p)
            s2 = self._cross(s_in, p_in, s_mask, p_mask,
                             self.q_proj_sp, self.k_proj_sp, self.v_proj_sp, self.out_sp)
            s = s + s2
        else:
            s2 = self._cross(s, p, s_mask, p_mask,
                             self.q_proj_sp, self.k_proj_sp, self.v_proj_sp, self.out_sp)
            s = self.s_norm2(s + s2)

        return p, s

######################
# [BÖLÜM 8: Tam Ablasyon Modeli]
######################
print("\n===== 6) Tam Ablasyon Modeli Tanımı =====")

class CrossAttentionFusionModelFull(nn.Module):
    """
    Tam parametreli DTA modeli:
    - pos_type: 'abs' | 'rope' | 'alibi' | 'none'
    - pre_norm + norm_type: Pre/Post × LN/RMS
    - ffn_type: 'relu' | 'gelu' | 'swiglu' | 'geglu'
    - attn_type: 'standard' | 'gated' | 'gqa' | 'none'
    """
    def __init__(self,
                 prot_vocab_size,
                 smi_vocab_size,
                 prot_max_len=1000,
                 smi_max_len=100,
                 d_model=256,
                 nhead=8,
                 num_encoder_layers=2,
                 num_cross_layers=1,
                 dim_feedforward=1024,
                 dropout=0.1,
                 padding_idx=0,
                 pos_type='abs',
                 pre_norm=False,
                 norm_type='ln',
                 ffn_type='relu',
                 attn_type='standard',
                 gqa_kv_heads=4):
        super().__init__()
        self.padding_idx = padding_idx
        self.pos_type = pos_type
        self.attn_type = attn_type

        # Token embeddings
        self.prot_token_emb = nn.Embedding(prot_vocab_size, d_model, padding_idx=padding_idx)
        self.smi_token_emb  = nn.Embedding(smi_vocab_size,  d_model, padding_idx=padding_idx)

        # Learnable absolute positional embedding sadece 'abs' için
        if pos_type == 'abs':
            self.prot_pos_emb = nn.Embedding(prot_max_len, d_model)
            self.smi_pos_emb  = nn.Embedding(smi_max_len,  d_model)
        else:
            self.prot_pos_emb = None
            self.smi_pos_emb  = None

        # attention içi PE: RoPE / ALiBi / none
        if pos_type in ['rope', 'alibi']:
            attn_pe = pos_type
        else:
            attn_pe = 'none'

        # Encoder katmanları
        self.prot_encoders = nn.ModuleList([
            TransformerEncoderBlockGeneral(
                d_model=d_model,
                nhead=nhead,
                dim_feedforward=dim_feedforward,
                dropout=dropout,
                ffn_type=ffn_type,
                pre_norm=pre_norm,
                norm_type=norm_type,
                attn_pe=attn_pe
            )
            for _ in range(num_encoder_layers)
        ])
        self.smi_encoders = nn.ModuleList([
            TransformerEncoderBlockGeneral(
                d_model=d_model,
                nhead=nhead,
                dim_feedforward=dim_feedforward,
                dropout=dropout,
                ffn_type=ffn_type,
                pre_norm=pre_norm,
                norm_type=norm_type,
                attn_pe=attn_pe
            )
            for _ in range(num_encoder_layers)
        ])

        # Cross-Attention katmanları
        self.cross_layers = nn.ModuleList()
        if attn_type != 'none':
            for _ in range(num_cross_layers):
                if attn_type == 'standard':
                    self.cross_layers.append(
                        BiDirectionalCrossAttention(
                            d_model=d_model,
                            nhead=nhead,
                            dropout=dropout,
                            attn_pe=attn_pe,
                            pre_norm=pre_norm,
                            norm_type=norm_type,
                            gated=False
                        )
                    )
                elif attn_type == 'gated':
                    self.cross_layers.append(
                        BiDirectionalCrossAttention(
                            d_model=d_model,
                            nhead=nhead,
                            dropout=dropout,
                            attn_pe=attn_pe,
                            pre_norm=pre_norm,
                            norm_type=norm_type,
                            gated=True
                        )
                    )
                elif attn_type == 'gqa':
                    self.cross_layers.append(
                        GroupedQueryCrossAttentionBlockGeneral(
                            d_model=d_model,
                            nhead=nhead,
                            n_kv_heads=gqa_kv_heads,
                            dropout=dropout,
                            attn_pe=attn_pe,
                            pre_norm=pre_norm,
                            norm_type=norm_type
                        )
                    )
                else:
                    raise ValueError(f"Bilinmeyen attn_type: {attn_type}")

        # Çıkış MLP
        self.fc = nn.Sequential(
            nn.Linear(d_model*2, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 1)
        )

    def forward(self, prot_in, smi_in):
        prot_mask = (prot_in == self.padding_idx)  # [B,Lp]
        smi_mask  = (smi_in  == self.padding_idx)  # [B,Ls]

        p_embed = self.prot_token_emb(prot_in)
        s_embed = self.smi_token_emb(smi_in)

        if self.pos_type == 'abs':
            p_pos = torch.arange(prot_in.size(1), device=prot_in.device).unsqueeze(0)
            s_pos = torch.arange(smi_in.size(1), device=smi_in.device).unsqueeze(0)
            p_embed = p_embed + self.prot_pos_emb(p_pos)
            s_embed = s_embed + self.smi_pos_emb(s_pos)

        # Encoderlar
        for enc in self.prot_encoders:
            p_embed = enc(p_embed, key_padding_mask=prot_mask)
        for enc in self.smi_encoders:
            s_embed = enc(s_embed, key_padding_mask=smi_mask)

        # Cross-Attention
        for cross in self.cross_layers:
            p_embed, s_embed = cross(p_embed, s_embed, p_mask=prot_mask, s_mask=smi_mask)

        # Maskeli mean pooling
        prot_mask_expanded = prot_mask.unsqueeze(-1).expand_as(p_embed)
        smi_mask_expanded  = smi_mask.unsqueeze(-1).expand_as(s_embed)

        p_sum = (p_embed * (~prot_mask_expanded)).sum(dim=1)
        p_cnt = (~prot_mask).sum(dim=1, keepdim=True).clamp(min=1)
        p_vec = p_sum / p_cnt

        s_sum = (s_embed * (~smi_mask_expanded)).sum(dim=1)
        s_cnt = (~smi_mask).sum(dim=1, keepdim=True).clamp(min=1)
        s_vec = s_sum / s_cnt

        out = self.fc(torch.cat([p_vec, s_vec], dim=1)).squeeze(-1)
        return out

######################
# [BÖLÜM 9: 4x4x4x4 Ablation Loop]
######################
print("\n===== 7) 4x4x4x4 Ablation Başlatılıyor =====")

positional_options = ['abs', 'rope', 'alibi', 'none']   # 4
norm_options = [
    ('post_ln',  dict(pre_norm=False, norm_type='ln')),
    ('post_rms', dict(pre_norm=False, norm_type='rms')),
    ('pre_ln',   dict(pre_norm=True,  norm_type='ln')),
    ('pre_rms',  dict(pre_norm=True,  norm_type='rms')),
]  # 4
ffn_options = ['relu', 'gelu', 'swiglu', 'geglu']       # 4
attn_options = ['standard', 'gated', 'gqa', 'none']     # 4

total_configs = len(positional_options) * len(norm_options) * len(ffn_options) * len(attn_options)
print(f"Toplam konfigürasyon sayısı: {total_configs}")

EPOCHS_PER_MODEL = 3  # runtime'a göre ayarla (ör. 2-5 arası)
LR = 1e-4

results = []
config_idx = 0
start_time = time.time()

for pos_type in positional_options:
    for norm_name, norm_cfg in norm_options:
        for ffn_type in ffn_options:
            for attn_type in attn_options:
                config_idx += 1
                print("\n" + "="*80)
                print(f"[Config {config_idx}/{total_configs}] pos={pos_type}, norm={norm_name}, "
                      f"ffn={ffn_type}, attn={attn_type}")
                print("="*80)

                set_seed(42)  # her model için aynı seed

                model = CrossAttentionFusionModelFull(
                    prot_vocab_size=len(protein_vocab),
                    smi_vocab_size=len(smiles_vocab),
                    prot_max_len=1000,
                    smi_max_len=100,
                    d_model=256,
                    nhead=8,
                    num_encoder_layers=2,
                    num_cross_layers=1,
                    dim_feedforward=1024,
                    dropout=0.1,
                    padding_idx=protein_vocab['<pad>'],
                    pos_type=pos_type,
                    pre_norm=norm_cfg['pre_norm'],
                    norm_type=norm_cfg['norm_type'],
                    ffn_type=ffn_type,
                    attn_type=attn_type,
                    gqa_kv_heads=4
                ).to(device)

                criterion = nn.MSELoss()
                optimizer = optim.Adam(model.parameters(), lr=LR)

                best_val_loss = float('inf')
                best_state = None

                for epoch in range(EPOCHS_PER_MODEL):
                    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
                    val_loss, val_preds, val_trues = evaluate(model, val_loader, criterion, device)
                    val_rmse = sqrt(mean_squared_error(val_trues, val_preds))

                    print(f"  Epoch {epoch+1}/{EPOCHS_PER_MODEL} "
                          f"- TrainLoss={train_loss:.4f}, ValLoss={val_loss:.4f}, ValRMSE={val_rmse:.4f}")

                    if val_loss < best_val_loss:
                        best_val_loss = val_loss
                        best_state = copy.deepcopy(model.state_dict())

                # En iyi state ile test
                if best_state is not None:
                    model.load_state_dict(best_state)

                test_loss, test_preds, test_trues = evaluate(model, test_loader, criterion, device)
                test_rmse = sqrt(mean_squared_error(test_trues, test_preds))
                test_ci   = concordance_index(test_trues, test_preds)
                test_rm2  = rm2_score(test_trues, test_preds)
                test_aupr = aupr_score(test_trues, test_preds, threshold=7.0)

                print(f"  >> Test MSE={test_loss:.4f}, RMSE={test_rmse:.4f}, "
                      f"CI={test_ci:.4f}, r_m^2={test_rm2:.4f}, AUPR={test_aupr:.4f}")

                results.append({
                    "config_idx": config_idx,
                    "pos_type": pos_type,
                    "norm": norm_name,
                    "pre_norm": norm_cfg['pre_norm'],
                    "norm_type": norm_cfg['norm_type'],
                    "ffn_type": ffn_type,
                    "attn_type": attn_type,
                    "epochs": EPOCHS_PER_MODEL,
                    "val_loss": best_val_loss,
                    "test_mse": float(test_loss),
                    "test_rmse": float(test_rmse),
                    "test_ci": float(test_ci),
                    "test_rm2": float(test_rm2),
                    "test_aupr": float(test_aupr),
                })

                # Sonuçları sürekli kaydet (iş yarıda kesilirse diye)
                results_df = pd.DataFrame(results)
                output_path = "/content/drive/MyDrive/TransformDTA/ablation_results.csv"
                results_df.to_csv(output_path, index=False)
                print("Kaydedilen yol:", output_path)

                # GPU belleğini temizle
                del model, optimizer, criterion
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

elapsed = (time.time() - start_time) / 3600.0
print(f"\n===== Ablasyon tamamlandı. Geçen süre: {elapsed:.2f} saat =====")
print("Sonuçlar 'ablation_results.csv' dosyasına yazıldı.")


===== 1) Ortam Kurulumu ve Paketler =====
Mounted at /content/drive
ERROR: Could not find a version that satisfies the requirement rdkit-pypi (from versions: none)
ERROR: No matching distribution found for rdkit-pypi
Kullanılan cihaz: cuda

===== 2) Veri Okuma ve İnceleme =====
DataFrame ilk 5 satır:
         ID        Target                                             SMILES  \
0  11314340          AAK1  CC1=C2C=C(C=CC2=NN1)C3=CC(=CN=C3)OCC(CC4=CC=CC...   
1  11314340   ABL1(E255K)  CC1=C2C=C(C=CC2=NN1)C3=CC(=CN=C3)OCC(CC4=CC=CC...   
2  11314340   ABL1(F317I)  CC1=C2C=C(C=CC2=NN1)C3=CC(=CN=C3)OCC(CC4=CC=CC...   
3  11314340  ABL1(F317I)p  CC1=C2C=C(C=CC2=NN1)C3=CC(=CN=C3)OCC(CC4=CC=CC...   
4  11314340   ABL1(F317L)  CC1=C2C=C(C=CC2=NN1)C3=CC(=CN=C3)OCC(CC4=CC=CC...   

                                            Sequence     Label  
0  MKKFFDSRREQGGSGLGSGSSGGGGSTSGLGSGYIGRVFGIGRQQV...  7.366532  
1  PFWKILNPLLERGTYYYFMGQQPGKVLGDQRRPSLPALHFIKGAGK...  5.000000  
2  PFWKILNPLLERGTYYYFM